# TFT Store-level Adaptation: End-to-End Notebook (L=28 → H=7)

- `pytorch-forecasting` 기본 경로. 실패 시 PyTorch LSTM fallback 자동 전환.

In [35]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

## 1) Install & Imports

In [36]:
# If your environment already has these, re-running is safe.
import os, sys, math, json, glob, gc, warnings, random
from dataclasses import dataclass, asdict
from typing import List, Dict, Tuple, Optional
import numpy as np
import pandas as pd

import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.loggers import CSVLogger

from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.models import TemporalFusionTransformer
from pytorch_forecasting.metrics import MAE, RMSE
from pytorch_forecasting.data import GroupNormalizer

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[env] device={device}, torch={torch.__version__}, pl={pl.__version__}")


[env] device=cuda, torch=2.8.0+cu128, pl=2.5.2


## 2) Config & Utils

In [37]:
# ----- Config -----
@dataclass
class Config:
    # Paths
    train_path: str = "./dataset/train.csv"
    test_glob: str = "./dataset/TEST_*.csv"
    sample_path: str = "./result/sample_submission_date.csv"  # as requested
    out_dir: str = "./artifacts_tft"
    adapters_dir: str = "./artifacts_tft/adapters"
    best_ckpt_path: str = "./artifacts_tft/best.ckpt"
    result_path: str = "./result/submission_tft_storeadapt.csv"
    log_csv_path: str = "./artifacts_tft/training_log.csv"

    # Windowing
    L: int = 28
    H: int = 7

    # K-fold CV
    K: int = 4
    embargo_days: int = 3

    # TFT hparams (sane defaults; edit freely)
    hidden_size: int = 128
    lstm_layers: int = 2
    attention_head_size: int = 4
    dropout: float = 0.1
    learning_rate: float = 3e-4
    batch_size: int = 128
    weight_decay: float = 1e-4
    loss_name: str = "MAE"  # "MAE" or "RMSE"

    # Training
    max_epochs_cv: int = 20
    max_epochs_full: int = 50
    patience: int = 5
    num_workers: int = 2
    precision: str = "16-mixed"  # use "32-true" if no AMP

    # Store adaptation (method B: affine)
    adapt_recent_anchors_per_store: int = 50
    adapt_decay_half_life: int = 14
    adapt_min_samples: int = 10

    # Reproducibility
    seed: int = 42

CFG = Config()

# Make dirs and save config
for p in [CFG.out_dir, CFG.adapters_dir, os.path.dirname(CFG.result_path), os.path.dirname(CFG.log_csv_path)]:
    if p and not os.path.exists(p):
        os.makedirs(p, exist_ok=True)
with open(os.path.join(CFG.out_dir, "config.json"), "w", encoding="utf-8") as f:
    json.dump(asdict(CFG), f, indent=2, ensure_ascii=False)

# ----- Repro -----
def set_seed(seed: int):
    pl.seed_everything(seed, workers=True)
    torch.backends.cudnn.benchmark = False
set_seed(CFG.seed)

# ----- Metrics -----
def smape_np(y_true, y_pred, eps: float = 1e-8):
    denom = (np.abs(y_true) + np.abs(y_pred) + eps)
    return 100.0 * np.mean(np.abs(y_pred - y_true) / denom)
def rmse_np(y_true, y_pred):
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

# ----- Preprocess -----
def ensure_store_menu(df: pd.DataFrame) -> pd.DataFrame:
    if "store_menu" not in df.columns:
        if "store_menu_id" in df.columns:
            df["store_menu"] = df["store_menu_id"].astype(str)
        else:
            assert "store" in df.columns and "menu" in df.columns, "Need columns: store, menu"
            df["store_menu"] = df["store"].astype(str) + "_" + df["menu"].astype(str)
    return df

def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df["date"]
    df["dow"] = d.dt.weekday
    df["is_weekend"] = (df["dow"] >= 5).astype(int)
    df["dom"] = d.dt.day
    df["month"] = d.dt.month
    df["is_eom"] = (d.dt.is_month_end).astype(int)
    df["is_payday25"] = (d.dt.day == 25).astype(int)

    df["dow_sin"]   = np.sin(2*np.pi*df["dow"]/7.0)
    df["dow_cos"]   = np.cos(2*np.pi*df["dow"]/7.0)
    df["month_sin"] = np.sin(2*np.pi*(df["month"]-1)/12.0)
    df["month_cos"] = np.cos(2*np.pi*(df["month"]-1)/12.0)
    df["dom_sin"]   = np.sin(2*np.pi*(df["dom"]-1)/31.0)
    df["dom_cos"]   = np.cos(2*np.pi*(df["dom"]-1)/31.0)
    return df

def fill_missing_dates(df: pd.DataFrame) -> pd.DataFrame:
    out = []
    for sm, g in df.groupby("store_menu"):
        idx = pd.date_range(g["date"].min(), g["date"].max(), freq="D")
        g2 = g.set_index("date").reindex(idx)
        g2.index.name = "date"
        g2 = g2.reset_index()
        g2["store_menu"] = sm
        for col in ["store","menu"]:
            if col in g.columns:
                g2[col] = g[col].iloc[0]
        if "sales" in g2.columns:
            g2["sales"] = g2["sales"].fillna(0)
        out.append(g2)
    return pd.concat(out, ignore_index=True)

def prep_core(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = ensure_store_menu(df)
    if "sales" in df.columns:
        df["sales"] = df["sales"].clip(lower=0)
    df = df.sort_values(["store_menu","date"]).reset_index(drop=True)
    df = fill_missing_dates(df)
    df = add_calendar_features(df)
    df["time_idx"] = df.groupby("store_menu").cumcount()
    if "store" in df.columns and "sales" in df.columns:
        df["store_total_sales"] = df.groupby(["store","date"])["sales"].transform("sum")
        df["store_total_ma7"] = df.groupby("store")["store_total_sales"].transform(lambda x: x.rolling(7, min_periods=1).mean())
    else:
        df["store_total_ma7"] = 0.0
    # ensure dtypes
    df["sales"] = pd.to_numeric(df["sales"], errors="coerce").fillna(0).clip(lower=0)
    cat_cols = ["store_menu", "dow"]
    if "store" in df.columns: cat_cols.append("store")
    if "menu"  in df.columns: cat_cols.append("menu")
    for c in cat_cols:
        df[c] = df[c].astype("category")
    return df

def choose_loss(name: str):
    if name.upper() == "RMSE":
        return RMSE()
    return MAE()


Seed set to 42


## 3) Dataset Builder + K-fold

In [38]:
def build_cv_anchors(df: pd.DataFrame, L: int, H: int, K: int, embargo_days: int) -> Dict[int, Dict[str, List[int]]]:
    # For each series, candidate anchors are L-1+embargo ... max_time_idx-H
    anchors_by_fold = {k: {} for k in range(K)}
    for sm, g in df.groupby("store_menu"):
        tmax = int(g["time_idx"].max())
        candidates = list(range(L-1 + embargo_days, tmax - H + 1))
        if len(candidates) == 0:
            continue
        splits = np.array_split(candidates, K)
        for k in range(K):
            anchors_by_fold[k][sm] = list(map(int, splits[k]))
    return anchors_by_fold

def mask_train_for_fold(df: pd.DataFrame, anchors_for_series: Dict[str, List[int]], H: int, embargo_days: int) -> pd.DataFrame:
    # Drop rows used as validation targets and embargo days just before anchors
    df = df.copy()
    keep = np.ones(len(df), dtype=bool)
    by_index = df.groupby("store_menu").indices
    for sm, anchors in anchors_for_series.items():
        idxs = by_index.get(sm, [])
        if len(idxs) == 0 or len(anchors) == 0:
            continue
        sub = df.iloc[idxs]
        bad_idx_set = set()
        for a in anchors:
            for t in range(a - embargo_days + 1, a + 1):   # embargo before anchor
                bad_idx_set.add(t)
            for t in range(a + 1, a + H + 1):              # validation targets
                bad_idx_set.add(t)
        keep[idxs] = ~sub["time_idx"].isin(bad_idx_set).values
    return df[keep].reset_index(drop=True)

def make_tsdataset(df: pd.DataFrame, L: int, H: int) -> TimeSeriesDataSet:
    static_categoricals = ["store","menu","store_menu"] if ("store" in df.columns and "menu" in df.columns) else ["store_menu"]
    time_varying_known_reals = ["dow_sin","dow_cos","month_sin","month_cos","dom_sin","dom_cos","is_weekend","is_eom","is_payday25"]
    time_varying_unknown_reals = ["sales","store_total_ma7"]
    time_varying_known_categoricals = ["dow"]

    return TimeSeriesDataSet(
        df,
        time_idx="time_idx",
        target="sales",
        group_ids=["store_menu"],
        max_encoder_length=L,
        max_prediction_length=H,
        time_varying_known_reals=time_varying_known_reals,
        time_varying_unknown_reals=time_varying_unknown_reals,
        static_categoricals=static_categoricals,
        time_varying_known_categoricals=time_varying_known_categoricals,
        target_normalizer=GroupNormalizer(groups=["store_menu"], transformation="softplus"),
        allow_missing_timesteps=True,   # ← 여기로 교체
    )


def try_build_loader(ds: TimeSeriesDataSet, batch_size: int, num_workers: int, shuffle: bool):
    bs = batch_size
    while bs >= 8:
        try:
            dl = ds.to_dataloader(train=shuffle, batch_size=bs, num_workers=num_workers)
            return dl, bs
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                bs = max(8, bs // 2)
                torch.cuda.empty_cache()
                print(f"[loader] CUDA OOM -> batch_size {bs}")
            else:
                raise
    return ds.to_dataloader(train=shuffle, batch_size=8, num_workers=num_workers), 8


## 4) TFT Wrapper

In [39]:
def build_tft(train_ds: TimeSeriesDataSet, cfg: Config) -> TemporalFusionTransformer:
    model = TemporalFusionTransformer.from_dataset(
    train_ds,
    hidden_size=CFG.hidden_size,
    lstm_layers=CFG.lstm_layers,
    attention_head_size=CFG.attention_head_size,
    dropout=CFG.dropout,
    learning_rate=CFG.learning_rate,
    loss=choose_loss(CFG.loss_name),
    reduce_on_plateau_patience=4,
    output_size=1  # 선택
    )

    print(f"[tft] params={sum(p.numel() for p in model.parameters()):,}")
    return model


## 5) Global 학습 + K-fold 평가

In [40]:
def predict_on_anchors(model: TemporalFusionTransformer, base_ds: TimeSeriesDataSet, df_all: pd.DataFrame,
                       anchors_for_series: Dict[str, List[int]], L: int, H: int) -> Tuple[np.ndarray, np.ndarray]:
    y_true_all, y_pred_all = [], []
    req_rows, meta = [], []
    for sm, g in df_all.groupby("store_menu"):
        g = g.sort_values("time_idx")
        t_to_row = {int(t): i for i, t in enumerate(g["time_idx"].values)}
        for a in anchors_for_series.get(sm, []):
            start = a - (L - 1)
            if start < 0: continue
            idxs = [t_to_row.get(t) for t in range(start, a+1)]
            if any(i is None for i in idxs): continue
            req_rows.append(g.iloc[idxs].copy()); meta.append((sm, a))
    if not req_rows:
        return np.array([]), np.array([])
    req_df = pd.concat(req_rows, ignore_index=True)
    pred_ds = TimeSeriesDataSet.from_dataset(base_ds, req_df, stop_randomization=True)
    pred_dl = pred_ds.to_dataloader(train=False, batch_size=CFG.batch_size, num_workers=CFG.num_workers)
    raw = model.predict(pred_dl, return_x=True, trainer_kwargs=dict(accelerator="gpu" if device=="cuda" else "cpu"))
    preds = raw.output
    for i, (sm, a) in enumerate(meta):
        g = df_all[df_all["store_menu"]==sm].sort_values("time_idx")
        tgt = g[(g["time_idx"]>=a+1) & (g["time_idx"]<=a+H)]["sales"].values
        if len(tgt)==H:
            y_true_all.append(tgt.astype(float))
            y_pred_all.append(preds[i].detach().cpu().numpy().astype(float))
    if not y_true_all:
        return np.array([]), np.array([])
    return np.vstack(y_true_all), np.vstack(y_pred_all)

def train_one_fold(df_fold_train: pd.DataFrame, df_all: pd.DataFrame,
                   anchors_for_series: Dict[str, List[int]], cfg: Config, fold_idx: int):
    # small temporal val for early stopping: last H+1 per series
    cutoff = df_fold_train.groupby("store_menu")["time_idx"].max().reset_index()
    cutoff["val_start"] = cutoff["time_idx"] - (cfg.H + 1)
    val_mask = pd.Series(False, index=df_fold_train.index)
    idx_map = df_fold_train.groupby("store_menu").indices
    for _, row in cutoff.iterrows():
        sm = row["store_menu"]; vs = int(row["val_start"])
        idxs = idx_map[sm]
        sub = df_fold_train.iloc[idxs]
        val_mask.iloc[idxs] = (sub["time_idx"] > vs).values
    df_train_inner = df_fold_train[~val_mask].reset_index(drop=True)
    df_val_inner   = df_fold_train[val_mask].reset_index(drop=True)

    ds_train = make_tsdataset(df_train_inner, cfg.L, cfg.H)
    ds_val   = TimeSeriesDataSet.from_dataset(ds_train, df_val_inner, stop_randomization=True)
    train_loader, bs_used = try_build_loader(ds_train, cfg.batch_size, cfg.num_workers, shuffle=True)
    val_loader,   _       = try_build_loader(ds_val,   cfg.batch_size, cfg.num_workers, shuffle=False)

    model = build_tft(ds_train, cfg)
    logger = CSVLogger(save_dir=cfg.out_dir, name=f"fold_{fold_idx}")
    ckpt_cb = ModelCheckpoint(monitor="val_loss", save_top_k=1, mode="min")
    es_cb = EarlyStopping(monitor="val_loss", patience=cfg.patience, mode="min")
    lr_cb = LearningRateMonitor(logging_interval="epoch")

    trainer = pl.Trainer(
        accelerator="gpu" if device=="cuda" else "cpu",
        devices=1,
        max_epochs=cfg.max_epochs_cv,
        precision=cfg.precision,
        logger=logger,
        callbacks=[ckpt_cb, es_cb, lr_cb],
        enable_progress_bar=True,
        deterministic=True,
        log_every_n_steps=50
    )

    trained=False; cur_bs=bs_used
    while not trained:
        try:
            trainer.fit(model, train_loader, val_loader)
            trained=True
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and cur_bs>8:
                cur_bs = max(8, cur_bs//2)
                print(f"[train] OOM fold {fold_idx} -> batch_size {cur_bs}")
                train_loader, cur_bs = try_build_loader(ds_train, cur_bs, cfg.num_workers, shuffle=True)
                val_loader, _       = try_build_loader(ds_val,   cur_bs, cfg.num_workers, shuffle=False)
                torch.cuda.empty_cache()
            else:
                raise

    best_model = TemporalFusionTransformer.load_from_checkpoint(ckpt_cb.best_model_path)
    y_true, y_pred = predict_on_anchors(best_model, ds_train, df_all, anchors_for_series, cfg.L, cfg.H)
    if y_true.size == 0:
        return dict(fold=fold_idx, smape=np.nan, rmse=np.nan, ckpt=ckpt_cb.best_model_path)
    smape = smape_np(y_true, y_pred); rmse = rmse_np(y_true, y_pred)
    print(f"[fold {fold_idx}] sMAPE={smape:.3f} RMSE={rmse:.3f} n={y_true.size}")
    return dict(fold=fold_idx, smape=smape, rmse=rmse, ckpt=ckpt_cb.best_model_path)

def run_cross_validation(df_all: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    anchors_folds = build_cv_anchors(df_all, cfg.L, cfg.H, cfg.K, cfg.embargo_days)
    logs = []
    for k in range(cfg.K):
        print(f"\n==== Fold {k}/{cfg.K-1} ====")
        df_train_k = mask_train_for_fold(df_all, anchors_folds[k], cfg.H, cfg.embargo_days)
        res = train_one_fold(df_train_k, df_all, anchors_folds[k], cfg, k)
        logs.append(res); gc.collect(); torch.cuda.empty_cache()
    log_df = pd.DataFrame(logs)
    log_df.to_csv(CFG.log_csv_path, index=False, encoding="utf-8-sig")
    print("\n[CV] Mean sMAPE:", np.nanmean(log_df["smape"].values))
    print("[CV] Mean RMSE:",  np.nanmean(log_df["rmse"].values))
    return log_df

def train_full_and_save_best(df_all: pd.DataFrame, cfg: Config) -> str:
    # temporal val near end for early stop
    cutoff = df_all.groupby("store_menu")["time_idx"].max().reset_index()
    cutoff["val_start"] = cutoff["time_idx"] - (cfg.H + 1)
    val_mask = pd.Series(False, index=df_all.index)
    idx_map = df_all.groupby("store_menu").indices
    for _, row in cutoff.iterrows():
        sm = row["store_menu"]; vs = int(row["val_start"])
        idxs = idx_map[sm]
        sub = df_all.iloc[idxs]
        val_mask.iloc[idxs] = (sub["time_idx"] > vs).values
    df_train_inner = df_all[~val_mask].reset_index(drop=True)
    df_val_inner   = df_all[val_mask].reset_index(drop=True)

    ds_train = make_tsdataset(df_train_inner, CFG.L, CFG.H)
    ds_val   = TimeSeriesDataSet.from_dataset(ds_train, df_val_inner, stop_randomization=True)
    train_loader, bs_used = try_build_loader(ds_train, CFG.batch_size, CFG.num_workers, shuffle=True)
    val_loader,   _       = try_build_loader(ds_val,   CFG.batch_size, CFG.num_workers, shuffle=False)

    model = build_tft(ds_train, CFG)
    ckpt_cb = ModelCheckpoint(dirpath=CFG.out_dir, filename="best", monitor="val_loss", mode="min", save_top_k=1)
    es_cb = EarlyStopping(monitor="val_loss", patience=CFG.patience, mode="min")
    lr_cb = LearningRateMonitor(logging_interval="epoch")
    logger = CSVLogger(save_dir=CFG.out_dir, name="full_train")

    trainer = pl.Trainer(
        accelerator="gpu" if device=="cuda" else "cpu",
        devices=1,
        max_epochs=CFG.max_epochs_full,
        precision=CFG.precision,
        callbacks=[ckpt_cb, es_cb, lr_cb],
        logger=logger,
        enable_progress_bar=True,
        deterministic=True,
        log_every_n_steps=50
    )

    trained=False; cur_bs=bs_used
    while not trained:
        try:
            trainer.fit(model, train_loader, val_loader)
            trained=True
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and cur_bs>8:
                cur_bs = max(8, cur_bs//2)
                print(f"[full] OOM -> batch_size {cur_bs}")
                train_loader, cur_bs = try_build_loader(ds_train, cur_bs, CFG.num_workers, shuffle=True)
                val_loader, _       = try_build_loader(ds_val,   cur_bs, CFG.num_workers, shuffle=False)
                torch.cuda.empty_cache()
            else:
                raise

    if os.path.exists(ckpt_cb.best_model_path):
        import shutil
        shutil.copy2(ckpt_cb.best_model_path, CFG.best_ckpt_path)
    print(f"[global] best ckpt -> {CFG.best_ckpt_path}")
    return CFG.best_ckpt_path


## 6) Store-level adaptation (방법 B: affine)

In [41]:
def build_anchor_windows_for_store(model: TemporalFusionTransformer, base_ds: TimeSeriesDataSet,
                                   df_all: pd.DataFrame, store: str, L: int, H: int, max_anchors: int) -> Tuple[np.ndarray, np.ndarray]:
    y_true_all, y_pred_all = [], []
    store_g = df_all[df_all["store"]==store] if "store" in df_all.columns else df_all.copy()
    for sm, g in store_g.groupby("store_menu"):
        g = g.sort_values("time_idx")
        tmax = int(g["time_idx"].max())
        candidates = list(range(L-1 + CFG.embargo_days, tmax - H + 1))
        if not candidates:
            continue
        anchors = candidates[-max_anchors:]
        req_rows, meta = [], []
        t_to_row = {int(t): i for i, t in enumerate(g["time_idx"].values)}
        for a in anchors:
            start = a - (L - 1)
            if start < 0: continue
            idxs = [t_to_row.get(t) for t in range(start, a+1)]
            if any(i is None for i in idxs): continue
            req_rows.append(g.iloc[idxs].copy()); meta.append((sm, a))
        if not req_rows:
            continue
        req_df = pd.concat(req_rows, ignore_index=True)
        pred_ds = TimeSeriesDataSet.from_dataset(base_ds, req_df, stop_randomization=True)
        pred_dl = pred_ds.to_dataloader(train=False, batch_size=CFG.batch_size, num_workers=CFG.num_workers)
        raw = model.predict(pred_dl, return_x=False, trainer_kwargs=dict(accelerator="gpu" if device=="cuda" else "cpu"))
        preds = raw
        for i, (_, a) in enumerate(meta):
            tgt = g[(g["time_idx"]>=a+1) & (g["time_idx"]<=a+H)]["sales"].values
            if len(tgt)==H:
                y_true_all.append(tgt.astype(float))
                y_pred_all.append(preds[i].detach().cpu().numpy().astype(float))
    if not y_true_all:
        return np.array([]), np.array([])
    return np.concatenate(y_pred_all).ravel(), np.concatenate(y_true_all).ravel()

def fit_store_adapter_wls(y_pred: np.ndarray, y_true: np.ndarray, half_life_days: int, H: int):
    n = len(y_pred)
    if n < 2:
        return 1.0, 0.0
    m = n // H
    ws = []
    for i in range(m):
        age = (m-1 - i)          # 0 for newest
        w = 0.5 ** (age / max(1, half_life_days))
        ws.extend([w]*H)
    W = np.diag(ws)
    X = np.vstack([y_pred, np.ones_like(y_pred)]).T
    y = y_true.reshape(-1,1)
    try:
        beta = np.linalg.pinv(X.T @ W @ X) @ (X.T @ W @ y)  # [a,b]
        a = float(beta[0,0]); b = float(beta[1,0])
    except np.linalg.LinAlgError:
        a, b = 1.0, 0.0
    return a, b

def run_store_adaptation(global_ckpt_path: str, df_all: pd.DataFrame, cfg: Config):
    base_ds = make_tsdataset(df_all, cfg.L, cfg.H)
    model = TemporalFusionTransformer.load_from_checkpoint(global_ckpt_path).to(device)
    stores = sorted(df_all["store"].dropna().unique()) if "store" in df_all.columns else ["global"]
    os.makedirs(cfg.adapters_dir, exist_ok=True)
    logs = []
    for store in stores:
        y_pred, y_true = build_anchor_windows_for_store(model, base_ds, df_all, store,
                                                        cfg.L, cfg.H, cfg.adapt_recent_anchors_per_store)
        if len(y_true) < cfg.adapt_min_samples:
            a, b = 1.0, 0.0
            smape_val = np.nan; rmse_val = np.nan
        else:
            a, b = fit_store_adapter_wls(y_pred, y_true, cfg.adapt_decay_half_life, cfg.H)
            y_adj = a*y_pred + b
            smape_val = smape_np(y_true, y_adj); rmse_val = rmse_np(y_true, y_adj)
        sdir = os.path.join(cfg.adapters_dir, str(store)); os.makedirs(sdir, exist_ok=True)
        with open(os.path.join(sdir, "adapter.json"), "w", encoding="utf-8") as f:
            json.dump({"a":a, "b":b, "metrics":{"sMAPE":smape_val, "RMSE":rmse_val}}, f, indent=2, ensure_ascii=False)
        logs.append(dict(store=store, a=a, b=b, smape=smape_val, rmse=rmse_val))
        print(f"[adapt] store={store} a={a:.4f} b={b:.4f} sMAPE={smape_val} RMSE={rmse_val}")
    adf = pd.DataFrame(logs)
    if os.path.exists(CFG.log_csv_path):
        prev = pd.read_csv(CFG.log_csv_path)
        pd.concat([prev, adf], ignore_index=True).to_csv(CFG.log_csv_path, index=False, encoding="utf-8-sig")
    else:
        adf.to_csv(CFG.log_csv_path, index=False, encoding="utf-8-sig")


## 7) Inference & Submission

In [42]:
def load_store_adapter(store: str, adapters_dir: str) -> Tuple[float, float]:
    path = os.path.join(adapters_dir, str(store), "adapter.json")
    if not os.path.exists(path):
        return 1.0, 0.0
    with open(path, "r", encoding="utf-8") as f:
        d = json.load(f)
    return float(d.get("a", 1.0)), float(d.get("b", 0.0))

def predict_for_testfile(model: TemporalFusionTransformer, base_ds: TimeSeriesDataSet, df_test_raw: pd.DataFrame,
                         cfg: Config) -> pd.DataFrame:
    df_test = prep_core(df_test_raw)
    pred_ds = TimeSeriesDataSet.from_dataset(base_ds, df_test, predict=True, stop_randomization=True)
    pred_dl = pred_ds.to_dataloader(train=False, batch_size=cfg.batch_size, num_workers=cfg.num_workers)
    raw = model.predict(pred_dl, return_x=True, trainer_kwargs=dict(accelerator="gpu" if device=="cuda" else "cpu"))
    preds = raw.output.detach().cpu().numpy()  # (groups, H)
    rows = []
    for i, (sm, g) in enumerate(df_test.groupby("store_menu")):
        store = g["store"].iloc[0] if "store" in g.columns else "global"
        last_date = g["date"].max()
        future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=cfg.H, freq="D")
        a, b = load_store_adapter(store, cfg.adapters_dir)
        y_adj = np.maximum(0.0, a*preds[i] + b)
        for d, v in zip(future_dates, y_adj):
            rows.append(dict(store_menu=sm, date=d.strftime("%Y-%m-%d"), sales=float(v)))
    return pd.DataFrame(rows)

def run_inference_and_submission(global_ckpt_path: str, train_df: pd.DataFrame, cfg: Config):
    base_ds = make_tsdataset(train_df, cfg.L, cfg.H)
    model = TemporalFusionTransformer.load_from_checkpoint(global_ckpt_path).to(device).eval()

    test_files = sorted(glob.glob(cfg.test_glob))
    all_preds = []
    for tf in test_files:
        print(f"[infer] {os.path.basename(tf)}")
        df_test_raw = pd.read_csv(tf)
        preds_df = predict_for_testfile(model, base_ds, df_test_raw, cfg)
        all_preds.append(preds_df)
    if not all_preds:
        raise FileNotFoundError("No test files matched.")
    pred_df = pd.concat(all_preds, ignore_index=True)

    sample = pd.read_csv(cfg.sample_path)
    if 'date' in sample.columns:
        dates = pd.to_datetime(sample['date'])
        sm_cols = [c for c in sample.columns if c != 'date']
        sub = pd.DataFrame({'date': dates})
    else:
        dates = sorted(pred_df['date'].unique())
        sm_cols = list(sample.columns)
        sub = pd.DataFrame({'date': dates})

    pivot = pred_df.pivot(index='date', columns='store_menu', values='sales').reindex(sub['date'].dt.strftime('%Y-%m-%d'))
    for c in sm_cols:
        if c not in pivot.columns:
            pivot[c] = 0.0
    pivot = pivot[sm_cols]
    pivot.reset_index(drop=False).rename(columns={'index':'date'}).to_csv(CFG.result_path, index=False, encoding='utf-8-sig')
    print(f"[submit] saved -> {CFG.result_path}")


## 8) Run-all

In [43]:
if __name__ == "__main__":
    print("[step] Load & preprocess train")
    assert os.path.exists(CFG.train_path), f"Train not found: {CFG.train_path}"
    train_df_raw = pd.read_csv(CFG.train_path)
    train_df = prep_core(train_df_raw)
    assert train_df['sales'].ge(0).all(), "Negative sales after clipping?"

    print("[step] Cross-validation")
    cv_log = run_cross_validation(train_df, CFG)

    print("[step] Train global on full train")
    best_ckpt = train_full_and_save_best(train_df, CFG)

    print("[step] Store adaptation")
    run_store_adaptation(best_ckpt, train_df, CFG)

    print("[step] Inference & submission")
    run_inference_and_submission(best_ckpt, train_df, CFG)

    print("[done] Pipeline completed.")


[step] Load & preprocess train
[step] Cross-validation

==== Fold 0/3 ====


ValueError: Data type of category dow was found to be numeric - use a string type / categorified string